# Kvasir-VQA x1 — Image-only baseline (frozen ViT + Logistic Regression)

Compute image embeddings using a frozen ViT backbone, then train a shallow classifier on top-K answers.

Outputs saved to `2_modeling/02_image_only/out/`.


In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, ViTModel

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler


2026-01-03 22:30:03.255603: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-03 22:30:03.255660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-03 22:30:03.256784: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-03 22:30:03.261724: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-03 22:30:04.175800: W tensorflow/compiler/tf2

In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "02_image_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 16
NUM_WORKERS = 0
TOP_K = 200
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out
Device: cuda


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)

# Resolve image paths relative to dataset root if needed
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Top-K answers from train split
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})


Top-K answers: 200
{'train': 46598, 'val': 5884, 'test': 5893}


In [5]:
# Image embedding extraction
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    images = [b[1] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return ids, inputs["pixel_values"]


def compute_embeddings(unique_df):
    ds = ImageDS(unique_df)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ViT embed"):
            pixels = pixels.to(DEVICE)
            out = vit(pixels).last_hidden_state[:, 0, :]
            for i, img_id in enumerate(ids):
                emb_map[img_id] = out[i].cpu().numpy()
    return emb_map


In [6]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
vit = ViTModel.from_pretrained(MODEL_NAME).to(DEVICE)
vit.eval()

# Compute or load cached embeddings
EMB_PATH = OUT_DIR / "image_embeddings.npz"

# Use unique images from filtered dataset
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    img_ids = data["img_ids"].tolist()
    embeddings = data["embeddings"]
    emb_map = {img_id: embeddings[i] for i, img_id in enumerate(img_ids)}
    print("Loaded cached embeddings:", len(emb_map))
else:
    emb_map = compute_embeddings(unique_imgs)
    img_ids = list(emb_map.keys())
    embeddings = np.stack([emb_map[i] for i in img_ids])
    np.savez(EMB_PATH, img_ids=np.array(img_ids), embeddings=embeddings)
    print("Saved embeddings:", EMB_PATH)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


ViT embed:   0%|          | 0/407 [00:00<?, ?it/s]

Saved embeddings: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out/image_embeddings.npz


In [7]:
# Build feature matrices

def build_X(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_train = build_X(train_k)
X_val = build_X(val_k) if len(val_k) else None
X_test = build_X(test_k) if len(test_k) else None

y_train = train_k["answer_norm"].values
y_val = val_k["answer_norm"].values if len(val_k) else None
y_test = test_k["answer_norm"].values if len(test_k) else None

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
if X_val is not None:
    X_val = scaler.transform(X_val)
if X_test is not None:
    X_test = scaler.transform(X_test)


In [ ]:
# Train classifier
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)


def eval_split(X, y_true, split_name):
    y_pred = clf.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    pred_df.to_csv(OUT_DIR / f"pred_{split_name}.csv", index=False)
    with open(OUT_DIR / f"metrics_{split_name}.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)
    print(split_name, metrics)


eval_split(X_train, y_train, "train")
if X_val is not None:
    eval_split(X_val, y_val, "val")
if X_test is not None:
    eval_split(X_test, y_test, "test")


train {'accuracy': 0.28091334392033995, 'macro_f1': 0.016073895699941186}
val {'accuracy': 0.21159075458871515, 'macro_f1': 0.01724070442155917}
test {'accuracy': 0.21839470558289495, 'macro_f1': 0.017799553540167993}


: 